# Extreme neerslag in Belgie: jaarlijkse maxima over 10 dagen

Deze notebook analyseert de jaarlijkse maximale neerslag over een periode van 10 opeenvolgende dagen. De focus ligt op drie vragen:

1. Hoe evolueert de volledige meetreeks doorheen de tijd?
2. Is er sinds 1981 een recente lineaire trend, zoals in de KMI-klimaatrapportage vaak wordt gebruikt?
3. Ligt het niveau vanaf 1981 duidelijk anders dan in de periode ervoor?

De analyse gebruikt de KMI-logica als leidraad: een gladde LOESS-curve voor de volledige reeks en een aparte recente trend vanaf 1981. Omdat extreme neerslag veel natuurlijke variabiliteit bevat, rapporteren we naast de schattingen ook robuuste onzekerheid en gevoeligheidschecks.


## 1. Setup en data

De ruwe Excel wordt, indien aanwezig, omgezet naar een compacte CSV. Daarna werkt de rest van de notebook uitsluitend met `processed_max_neerslag.csv`, zodat de analyse reproduceerbaar en sneller uitvoerbaar blijft.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
DATA_DIR = next((p / 'data' for p in PROJECT_ROOT_CANDIDATES if (p / 'data').exists()), Path('../data'))
CSV_PATH = DATA_DIR / 'processed_max_neerslag.csv'
EXCEL_CANDIDATES = [
    Path('../../Maximale neerslag.xlsx'),
    Path('../Maximale neerslag.xlsx'),
    Path('Maximale neerslag.xlsx'),
]

OBS_COL = 'Waarneming 10 dagen'
TREND_COL = 'Trendlijn 10 dagen'
START_RECENT = 1981
LOESS_FRAC = 0.25

plt.rcParams.update({
    'figure.figsize': (11, 6),
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.titleweight': 'bold',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.frameon': False,
})

COLORS = {
    'observed': '#4B5563',
    'loess': '#2563EB',
    'recent': '#DC2626',
    'accent': '#059669',
    'neutral': '#9CA3AF',
}


def create_processed_csv_from_excel(output_path=CSV_PATH):
    excel_path = next((path for path in EXCEL_CANDIDATES if path.exists()), None)
    if excel_path is None:
        raise FileNotFoundError(
            'Geen processed CSV en geen Excel-bestand gevonden. '
            'Plaats Maximale neerslag.xlsx in de projectmap of voer de datavoorbereiding uit.'
        )

    raw = pd.read_excel(excel_path, sheet_name='Data', engine='openpyxl')
    years = raw.iloc[2, 1:].values
    processed = pd.DataFrame({
        'Waarneming 1 uur': raw.iloc[9, 1:].values,
        'Waarneming 1 dag': raw.iloc[10, 1:].values,
        'Waarneming 10 dagen': raw.iloc[11, 1:].values,
        'Trendlijn 1 uur': raw.iloc[12, 1:].values,
        'Trendlijn 1 dag': raw.iloc[13, 1:].values,
        'Trendlijn 10 dagen': raw.iloc[14, 1:].values,
    }, index=years)
    processed.index.name = 'Jaar'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    processed.to_csv(output_path)
    return output_path


def load_extreme_series(csv_path=CSV_PATH):
    if not csv_path.exists():
        create_processed_csv_from_excel(csv_path)

    df = pd.read_csv(csv_path)
    extreme = (
        df[['Jaar', OBS_COL]]
        .dropna()
        .rename(columns={OBS_COL: 'max10d_mm'})
        .assign(Jaar=lambda x: x['Jaar'].astype(int))
        .sort_values('Jaar')
        .reset_index(drop=True)
    )
    return df, extreme


def hac_lags(n):
    return max(1, int(np.floor(4 * (n / 100) ** (2 / 9))))


def format_p(value):
    if pd.isna(value):
        return ''
    return '<0,001' if value < 0.001 else f'{value:.3f}'.replace('.', ',')


df, extreme = load_extreme_series()
extreme.head()


## 2. Beschrijving van de reeks

De reeks bevat jaarlijkse maxima. Elk punt is dus het hoogste 10-daagse neerslagtotaal binnen een jaar, niet het jaargemiddelde. Daardoor is de variabiliteit groot en moeten trends voorzichtig worden geinterpreteerd.


In [ ]:
years = extreme['Jaar'].to_numpy(dtype=float)
values = extreme['max10d_mm'].to_numpy(dtype=float)

summary = pd.DataFrame({
    'Kenmerk': [
        'Aantal jaren',
        'Periode',
        'Gemiddelde',
        'Mediaan',
        'Standaardafwijking',
        'Maximum',
        'Jaar van maximum',
    ],
    'Waarde': [
        len(extreme),
        f'{int(years.min())}-{int(years.max())}',
        f'{values.mean():.1f} mm',
        f'{np.median(values):.1f} mm',
        f'{values.std(ddof=1):.1f} mm',
        f'{values.max():.1f} mm',
        int(years[values.argmax()]),
    ],
})

display(summary)

fig, ax = plt.subplots()
ax.plot(extreme['Jaar'], extreme['max10d_mm'], color=COLORS['observed'], linewidth=1.2, alpha=0.75)
ax.scatter(extreme['Jaar'], extreme['max10d_mm'], color=COLORS['observed'], s=22, alpha=0.70)
ax.set_title('Jaarlijkse maximale 10-daagse neerslag')
ax.set_xlabel('Jaar')
ax.set_ylabel('Neerslag (mm)')
ax.margins(x=0.01)
plt.tight_layout()
plt.show()


## 3. Volledige reeks: LOESS als beschrijvende trend

Voor de volledige periode gebruiken we LOESS. Die curve vat de vorm van de evolutie samen zonder een rechte lijn op te leggen aan de volledige meetperiode. Dit is een beschrijvende visualisatie en geen significantietest.


In [ ]:
loess_curve = lowess(values, years, frac=LOESS_FRAC, it=1, return_sorted=True)

fig, ax = plt.subplots()
ax.scatter(years, values, color=COLORS['observed'], alpha=0.55, s=28, label='Jaarlijks maximum')
ax.plot(
    loess_curve[:, 0],
    loess_curve[:, 1],
    color=COLORS['loess'],
    linewidth=2.8,
    label=f'LOESS volledige reeks (frac={LOESS_FRAC})',
)
ax.set_title('Langetermijnevolutie met LOESS')
ax.set_xlabel('Jaar')
ax.set_ylabel('Neerslag (mm)')
ax.legend(loc='upper left')
ax.margins(x=0.01)
plt.tight_layout()
plt.show()


## 4. Recente trend vanaf 1981

De recente verandering wordt apart geschat met een lineaire trend vanaf 1981. De helling wordt uitgedrukt in mm per decennium. De onzekerheid gebruikt HAC/Newey-West standaardfouten, zodat de p-waarde minder gevoelig is voor autocorrelatie en heteroskedasticiteit in de residualen.


In [ ]:
recent = extreme.loc[extreme['Jaar'] >= START_RECENT].copy()
t_recent = recent['Jaar'].to_numpy(dtype=float) - START_RECENT
y_recent = recent['max10d_mm'].to_numpy(dtype=float)

ols_recent = sm.OLS(y_recent, sm.add_constant(t_recent)).fit()
recent_hac_lags = hac_lags(len(recent))
hac_recent = ols_recent.get_robustcov_results(cov_type='HAC', maxlags=recent_hac_lags)

slope_year = hac_recent.params[1]
slope_decade = slope_year * 10
ci90 = hac_recent.conf_int(alpha=0.10)[1] * 10
ci95 = hac_recent.conf_int(alpha=0.05)[1] * 10
p_hac = hac_recent.pvalues[1]

sen = stats.theilslopes(y_recent, recent['Jaar'].to_numpy(dtype=float), alpha=0.90)


def mann_kendall_test(series):
    series = np.asarray(series, dtype=float)
    n = len(series)
    s = 0.0
    for k in range(n - 1):
        s += np.sign(series[k + 1:] - series[k]).sum()

    _, counts = np.unique(series, return_counts=True)
    tie_term = np.sum(counts * (counts - 1) * (2 * counts + 5))
    var_s = (n * (n - 1) * (2 * n + 5) - tie_term) / 18
    z = (s - 1) / np.sqrt(var_s) if s > 0 else (s + 1) / np.sqrt(var_s) if s < 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    tau = s / (0.5 * n * (n - 1))
    return tau, p_value


mk_tau, mk_p = mann_kendall_test(y_recent)
trend_table = pd.DataFrame({
    'Methode': [
        'Lineaire trend met HAC-onzekerheid',
        'Theil-Sen robuuste helling',
        'Mann-Kendall monotone trend',
    ],
    'Schatting': [
        f'{slope_decade:.2f} mm/decennium',
        f'{sen.slope * 10:.2f} mm/decennium',
        f'tau = {mk_tau:.3f}',
    ],
    'Onzekerheid / toets': [
        f'90% BI {ci90[0]:.2f} tot {ci90[1]:.2f}; p = {format_p(p_hac)}',
        f'90% BI {sen.low_slope * 10:.2f} tot {sen.high_slope * 10:.2f} mm/decennium',
        f'p = {format_p(mk_p)}',
    ],
})

display(trend_table)

recent_fit = hac_recent.params[0] + hac_recent.params[1] * t_recent
significance = 'significant op 95%' if p_hac < 0.05 else 'significant op 90%' if p_hac < 0.10 else 'niet significant op 90%'

fig, ax = plt.subplots()
ax.scatter(years, values, color=COLORS['observed'], alpha=0.48, s=28, label='Jaarlijks maximum')
ax.plot(loess_curve[:, 0], loess_curve[:, 1], color=COLORS['loess'], linewidth=2.5, label='LOESS volledige reeks')
ax.plot(recent['Jaar'], recent_fit, color=COLORS['recent'], linewidth=2.4, label=f'Lineaire trend sinds {START_RECENT}')
ax.axvline(START_RECENT, color=COLORS['recent'], linestyle=':', linewidth=1.3)
ax.text(
    0.02,
    0.98,
    f'Trend sinds {START_RECENT}: {slope_decade:.2f} mm/decennium\nHAC p = {p_hac:.3f} ({significance})',
    transform=ax.transAxes,
    va='top',
    ha='left',
    fontsize=10,
    bbox={'facecolor': 'white', 'edgecolor': '#D1D5DB', 'alpha': 0.94, 'boxstyle': 'round,pad=0.35'},
)
ax.set_title('LOESS volledige reeks en recente trend vanaf 1981')
ax.set_xlabel('Jaar')
ax.set_ylabel('Neerslag (mm)')
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.12), ncol=3)
ax.margins(x=0.01)
plt.tight_layout()
plt.show()


## 5. Diagnostiek van de recente regressie

De recente trend is alleen bruikbaar als samenvatting wanneer de residualen niet sterk gestructureerd blijven. Daarom controleren we autocorrelatie expliciet met lag-1 autocorrelatie, Durbin-Watson en Ljung-Box tests.


In [ ]:
residuals_recent = pd.Series(ols_recent.resid, index=recent['Jaar'])
residual_diagnostics = pd.DataFrame({
    'Diagnose': [
        'Aantal jaren sinds 1981',
        'HAC maxlags',
        'Lag-1 autocorrelatie residualen',
        'Durbin-Watson residualen',
    ],
    'Waarde': [
        len(recent),
        recent_hac_lags,
        residuals_recent.autocorr(lag=1),
        durbin_watson(residuals_recent),
    ],
})

ljung_box = (
    acorr_ljungbox(residuals_recent, lags=[1, 5, 10], return_df=True)
    .rename(columns={'lb_stat': 'Ljung-Box statistic', 'lb_pvalue': 'p-waarde'})
)

display(residual_diagnostics.round(4))
display(ljung_box.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].axhline(0, color=COLORS['neutral'], linewidth=1)
axes[0].plot(residuals_recent.index, residuals_recent, color=COLORS['observed'], linewidth=1.4)
axes[0].scatter(residuals_recent.index, residuals_recent, color=COLORS['observed'], s=24)
axes[0].set_title('Residualen recente trend')
axes[0].set_xlabel('Jaar')
axes[0].set_ylabel('Residual (mm)')

axes[1].hist(residuals_recent, bins=12, color=COLORS['loess'], alpha=0.78, edgecolor='white')
axes[1].axvline(0, color=COLORS['neutral'], linewidth=1)
axes[1].set_title('Verdeling van residualen')
axes[1].set_xlabel('Residual (mm)')
axes[1].set_ylabel('Aantal jaren')
plt.tight_layout()
plt.show()


## 6. Niveauverschil vanaf 1981

Naast een lineaire trend testen we of de periode vanaf 1981 gemiddeld op een ander niveau ligt. Dat is een andere vraag dan de trendvraag: een hoger gemiddelde na 1981 betekent niet automatisch dat er precies in 1981 een abrupte klimaatsprong is geweest.


In [ ]:
level = extreme.copy()
level['post1981'] = (level['Jaar'] >= START_RECENT).astype(int)
level['t_center'] = level['Jaar'] - START_RECENT
level['post_t'] = level['post1981'] * level['t_center']

pre = level.loc[level['Jaar'] < START_RECENT, 'max10d_mm'].to_numpy(dtype=float)
post = level.loc[level['Jaar'] >= START_RECENT, 'max10d_mm'].to_numpy(dtype=float)

period_summary = pd.DataFrame({
    'Periode': ['voor 1981', 'vanaf 1981', 'verschil post-pre'],
    'n': [len(pre), len(post), ''],
    'Gemiddelde (mm)': [pre.mean(), post.mean(), post.mean() - pre.mean()],
    'Mediaan (mm)': [np.median(pre), np.median(post), np.median(post) - np.median(pre)],
    'Standaardafwijking (mm)': [pre.std(ddof=1), post.std(ddof=1), np.nan],
})

display(period_summary.round(2))


def fit_hac_level_model(columns, label):
    model = sm.OLS(level['max10d_mm'], sm.add_constant(level[columns])).fit()
    robust = model.get_robustcov_results(cov_type='HAC', maxlags=hac_lags(len(level)))
    rows = []
    for name, estimate, p_value, ci in zip(
        robust.model.exog_names,
        robust.params,
        robust.pvalues,
        robust.conf_int(alpha=0.05),
    ):
        rows.append({
            'Model': label,
            'Parameter': name,
            'Schatting': estimate,
            'CI95 laag': ci[0],
            'CI95 hoog': ci[1],
            'p-waarde': p_value,
        })
    return pd.DataFrame(rows)

level_models = pd.concat([
    fit_hac_level_model(['post1981'], 'Alleen niveauverschil'),
    fit_hac_level_model(['t_center', 'post1981'], 'Trend + niveauverschil'),
    fit_hac_level_model(['t_center', 'post1981', 'post_t'], 'Piecewise trend + niveauverschil'),
], ignore_index=True)

step_effects = level_models.loc[level_models['Parameter'].eq('post1981')].copy()
step_effects['p-waarde'] = step_effects['p-waarde'].map(format_p)
display(step_effects.round({'Schatting': 2, 'CI95 laag': 2, 'CI95 hoog': 2}))

fig, ax = plt.subplots(figsize=(8.5, 5.2))
parts = ax.violinplot([pre, post], showmeans=False, showmedians=False, widths=0.75)
for body in parts['bodies']:
    body.set_facecolor(COLORS['loess'])
    body.set_edgecolor('none')
    body.set_alpha(0.22)
for key in ['cbars', 'cmins', 'cmaxes']:
    parts[key].set_color(COLORS['neutral'])

ax.boxplot(
    [pre, post],
    widths=0.28,
    patch_artist=True,
    boxprops={'facecolor': 'white', 'color': COLORS['observed']},
    medianprops={'color': COLORS['recent'], 'linewidth': 2},
    whiskerprops={'color': COLORS['observed']},
    capprops={'color': COLORS['observed']},
    flierprops={'marker': 'o', 'markersize': 4, 'markerfacecolor': COLORS['observed'], 'markeredgecolor': 'white', 'alpha': 0.65},
)
ax.set_xticks([1, 2])
ax.set_xticklabels([f'Voor {START_RECENT}', f'Vanaf {START_RECENT}'])
ax.set_title('Verdeling voor en vanaf 1981')
ax.set_ylabel('Neerslag (mm)')
plt.tight_layout()
plt.show()


## 7. Conclusie

De volledige reeks **1898-2024** wordt het best beschreven met een LOESS-curve, omdat een globale rechte trend de niet-lineaire evolutie kan verbergen. Voor de recente periode **1981-2024** bedraagt de lineaire trend ongeveer **-1,47 mm per decennium**. Met HAC/Newey-West robuuste onzekerheid is die trend **niet significant** (`p ongeveer 0,43`) en ook de robuuste Theil-Sen en Mann-Kendall checks wijzen niet op een duidelijke monotone recente stijging.

De periode vanaf 1981 ligt wel hoger dan de periode ervoor: het gemiddelde verschil is ongeveer **+11,3 mm** en het mediaanverschil ongeveer **+9,2 mm**. Dat niveauverschil is duidelijk in een eenvoudige pre/post vergelijking. Zodra we tegelijk corrigeren voor een langetermijntrend, wordt het bewijs voor een zuivere abrupte level shift rond 1981 minder sterk.

Kort samengevat: de 10-daagse extreme neerslagreeks toont een hoger recent niveau, maar geen robuust significante stijgende lineaire trend sinds 1981. Voor uitspraken over terugkeerperioden of ontwerpwaarden is een afzonderlijke extreme-waardenanalyse, bijvoorbeeld met GEV-modellen, nodig.
